# CENG313 Introduction to Data Science — Assignment 1: EDA and Baseline
## CAFA 6 Protein Function Prediction

**Group:** GROUP47 &nbsp;|&nbsp; **Members:** Ekin Boran İBİLİ (23118080073), Berat DEMİR (23118080762)

**Competition:** [CAFA 6 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-6-protein-function-prediction)

**Contents**
0. Setup and data loading
1. Problem definition (task, target, metric)
2. Data overview
3. Exploratory data analysis
4. Preprocessing
5. Baseline models
6. Kaggle submission
7. Conclusion

Reusable code lives in `src/` (`data.py`, `models.py`, `utils.py`) and is imported here.

## 0. Setup and data loading

On Google Colab the next cell clones the project repository so that `src/` can be imported.

In [ ]:
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ceng313-2026/GROUP47_CENG313.git"

if IN_COLAB and not Path("src").exists():
    if not Path("project/src").exists():   # already cloned if the runtime was restarted
        cloned = subprocess.run(["git", "clone", REPO_URL, "project"], capture_output=True, text=True)
        if cloned.returncode != 0:
            raise RuntimeError(
                "Could not clone " + REPO_URL + " (wrong address or private repository).\n"
                "Either fix REPO_URL, or create a folder named 'src' in the Colab file panel, "
                "upload the files of src/ into it and run this cell again.\n" + cloned.stderr)
    os.chdir("project")

sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

### Getting the data

The dataset is **not stored in the repository**; it is downloaded from Kaggle with the Kaggle API into `data/` (ignored by git). You need to

1. join the competition on its Kaggle page and accept the rules (once per account), and
2. create an API key file on Kaggle (*Settings → API → Legacy API Credentials → Create Legacy API Key*), which downloads `kaggle.json`.

On Colab the cell below asks you to upload `kaggle.json`. If the data is already in `DATA_DIR`, nothing is downloaded.

In [ ]:
COMPETITION = "cafa-6-protein-function-prediction"
DATA_DIR = Path(os.environ.get("CAFA6_DATA_DIR", "data"))


def ensure_kaggle_credentials():
    token = Path.home() / ".kaggle" / "kaggle.json"
    if IN_COLAB and not token.exists():
        from google.colab import files
        uploaded = files.upload()  # choose kaggle.json
        token.parent.mkdir(exist_ok=True)
        token.write_bytes(uploaded["kaggle.json"])
        token.chmod(0o600)


def download_data():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    ensure_kaggle_credentials()
    subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION, "-p", str(DATA_DIR)], check=True)
    for archive in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DATA_DIR)
        archive.unlink()


if not list(DATA_DIR.rglob("train_terms.tsv")):
    download_data()

for path in sorted(p for p in DATA_DIR.rglob("*") if p.is_file()):
    print(f"{path.stat().st_size / 1e6:9.1f} MB  {path.relative_to(DATA_DIR)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.model_selection import train_test_split

from src.data import (AMINO_ACIDS, ASPECT_FULL_NAMES, ASPECTS, ROOT_TERMS, GeneOntology,
                      amino_acid_composition, build_label_matrix, load_cafa6, propagate_annotations)
from src.models import KmerFeaturizer, KNNBaseline, MLPBaseline, NaiveFrequencyBaseline
from src.utils import (ASPECT_COLORS, MODEL_STYLES, NEUTRAL, SPLIT_COLORS, evaluate_predictions,
                       format_count, propagate_scores, results_row, set_plot_style, write_submission)

SEED = 42
np.random.seed(SEED)
set_plot_style()
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
start = time.time()
data = load_cafa6(DATA_DIR)
train_seq, train_terms, train_tax = data["train_seq"], data["train_terms"], data["train_tax"]
test_seq, test_taxa, ia = data["test_seq"], data["test_taxa"], data["ia"]
go = GeneOntology.from_obo(data["obo_path"])
print(f"loaded in {time.time() - start:.1f} s")

## 1. Problem definition

### 1.1 Task

Proteins do most of the work in a cell, but for the large majority of known protein sequences the function has never been measured in a laboratory. CAFA (Critical Assessment of Functional Annotation) asks: **given only the amino-acid sequence of a protein (and its species), which functions does it have?**

- **Input:** a protein sequence, i.e. a string over 20 amino-acid letters (length from tens to thousands of residues).
- **Output:** for every protein, a list of Gene Ontology (GO) terms with a confidence score in (0, 1].
- **Type of problem:** supervised **multi-label classification** with a very large label space (tens of thousands of possible GO terms; each protein has several).

### 1.2 Target

The target is the set of GO terms of a protein. GO has three sub-ontologies (*aspects*), which are evaluated separately:

| Code | Aspect | Question it answers |
|---|---|---|
| MFO | Molecular Function | What does the protein do at the molecular level? |
| BPO | Biological Process | Which larger process does it take part in? |
| CCO | Cellular Component | Where in the cell does it act? |

Two properties make this target unusual:

1. **Hierarchy.** GO terms form a directed acyclic graph. A term implies all of its ancestors (a *protein kinase* is also a *kinase* and a *catalytic activity*).
2. **Open world.** A missing annotation means "not known yet", not "the protein does not have this function".

The cell below shows the target of one training protein.

In [ ]:
example_id = train_terms["EntryID"].iloc[0]
example_seq = train_seq.loc[train_seq["EntryID"] == example_id, "sequence"].iloc[0]
print(f"protein {example_id}: {len(example_seq)} amino acids")
print("sequence starts with:", example_seq[:70] + "...")
example_target = train_terms[train_terms["EntryID"] == example_id].copy()
example_target["term name"] = example_target["term"].map(go.name)
display(example_target[["aspect", "term", "term name"]].sort_values("aspect").reset_index(drop=True))

### 1.3 Metric

The competition uses the **protein-centric, information-accretion weighted F-max**, computed separately for MFO, BPO and CCO and then averaged.

Each GO term $f$ has a weight $IA(f) = -\log_2 P(f \mid \text{parents of } f)$, its *information accretion*: generic terms that almost always follow from their parents weigh close to 0, rare and specific terms weigh a lot. For a threshold $t$, protein $i$, its predicted term set $P_i(t)$ (terms with score $\ge t$) and its true term set $T_i$:

$$\text{pr}_i(t)=\frac{\sum_{f\in P_i(t)\cap T_i} IA(f)}{\sum_{f\in P_i(t)} IA(f)},\qquad
\text{rc}_i(t)=\frac{\sum_{f\in P_i(t)\cap T_i} IA(f)}{\sum_{f\in T_i} IA(f)}$$

Precision is averaged over the proteins with at least one prediction at $t$, recall over all proteins that have ground truth in the aspect, and

$$F_{\max}=\max_t \frac{2\,\text{pr}(t)\,\text{rc}(t)}{\text{pr}(t)+\text{rc}(t)}$$

So the score rewards predicting **specific** terms correctly and lets us choose the best threshold afterwards. Our implementation is `evaluate_predictions` in `src/utils.py`.

The competition is closed and the test labels are not public, so in this notebook all models are compared on a **held-out 20% split of the training data** (`random_state=42`) with this metric (see sections 4 and 6).

## 2. Data overview

### 2.1 Tables, shapes and data types

In [ ]:
tables = {
    "train_sequences.fasta": train_seq,
    "train_terms.tsv": train_terms,
    "train_taxonomy.tsv": train_tax,
    "IA.tsv": ia.rename("IA").reset_index(),
    "testsuperset.fasta": test_seq,
    "testsuperset-taxon-list.tsv": test_taxa,
}
shape_table = pd.DataFrame([
    {"file": name, "rows": format_count(len(frame)), "columns": frame.shape[1],
     "columns and dtypes": ", ".join(f"{col} ({dtype})" for col, dtype in frame.dtypes.astype(str).items())}
    for name, frame in tables.items()
])
display(shape_table)
print(f"go-basic.obo: {len(go) - len(go.obsolete):,} GO terms (plus {len(go.obsolete):,} obsolete)")

In [ ]:
display(train_seq.head(3))
display(train_terms.head(5))
display(train_tax.head(3))
display(ia.head(5).to_frame())
display(test_seq.head(3))

### 2.2 Summary statistics (`describe()`)

In [ ]:
display(pd.DataFrame({"train length": train_seq["length"].describe(), "test length": test_seq["length"].describe()}).T)
display(train_terms.astype(str).describe().T)
display(train_tax.astype(str).describe().T)
display(ia.describe().to_frame().T)

overview = pd.DataFrame(
    [
        ("training proteins", train_seq["EntryID"].nunique()),
        ("annotation rows (protein, term)", len(train_terms)),
        ("proteins with at least one annotation", train_terms["EntryID"].nunique()),
        ("distinct GO terms used in train", train_terms["term"].nunique()),
        ("distinct species in train", train_tax["taxonomyID"].nunique()),
        ("test proteins", test_seq["EntryID"].nunique()),
        ("distinct species in test", test_seq["taxonomyID"].nunique()),
    ],
    columns=["quantity", "value"],
)
overview["value"] = overview["value"].map(format_count)
display(overview)

### 2.3 Missing values

In [ ]:
missing_rows = []
for name, frame in tables.items():
    for column in frame.columns:
        values = frame[column]
        empty = int((values.astype(str).str.strip() == "").sum()) if values.dtype.kind in "OUT" or str(values.dtype).startswith(("str", "string")) else 0
        missing_rows.append({"file": name, "column": column, "missing (NaN)": int(values.isna().sum()), "empty strings": empty})
display(pd.DataFrame(missing_rows))

### 2.4 Duplicates and consistency between files

In [ ]:
standard = set(AMINO_ACIDS)
has_nonstandard = train_seq["sequence"].map(lambda s: not set(s) <= standard)
train_term_set = set(train_terms["term"])
labelled = set(train_terms["EntryID"])

quality = pd.DataFrame(
    [
        ("duplicate protein ids in train FASTA", train_seq["EntryID"].duplicated().sum()),
        ("duplicate protein ids in test FASTA", test_seq["EntryID"].duplicated().sum()),
        ("duplicate sequences in train (different ids, same sequence)", train_seq["sequence"].duplicated().sum()),
        ("duplicate annotation rows", train_terms.duplicated(["EntryID", "term"]).sum()),
        ("train proteins with a sequence but no annotation", (~train_seq["EntryID"].isin(labelled)).sum()),
        ("annotated proteins without a sequence", len(labelled - set(train_seq["EntryID"]))),
        ("train proteins without a taxon id", (~train_seq["EntryID"].isin(set(train_tax["EntryID"]))).sum()),
        ("train sequences with non-standard residues (X, U, B, Z, O)", has_nonstandard.sum()),
        ("annotated GO terms missing from the ontology file", sum(go.canonical(t) not in go for t in train_term_set)),
        ("annotated GO terms that are alternative ids", sum(t in go.alt_ids for t in train_term_set)),
        ("annotated GO terms that are obsolete", sum(go.canonical(t) in go.obsolete for t in train_term_set)),
        ("annotated GO terms without an IA weight", sum(t not in ia.index for t in train_term_set)),
        ("protein ids present in both train and test", len(set(train_seq["EntryID"]) & set(test_seq["EntryID"]))),
        ("empty sequences (train + test)", (train_seq["length"] == 0).sum() + (test_seq["length"] == 0).sum()),
    ],
    columns=["check", "count"],
)
display(quality)

**Interpretation — write after running:**
- Size of the data: number of proteins, annotation rows, distinct GO terms, species (use the numbers above).
- Are there missing values or duplicates? Which checks are non-zero and what do we do about them (see section 4)?
- If train and test share protein ids: in CAFA a protein that already has labels in one aspect can still be a prediction target for another aspect.

## 3. Exploratory data analysis

### 3.1 Sequence length (distribution and outliers)

Sequence length matters for feature design (k-mer counts grow with length) and later for protein language models, which truncate long inputs.

In [ ]:
all_lengths = pd.concat([train_seq["length"], test_seq["length"]])
bins = np.logspace(np.log10(max(all_lengths.min(), 1)), np.log10(all_lengths.max() + 1), 60)

fig, ax = plt.subplots(figsize=(8, 3.6))
for split, frame in [("train", train_seq), ("test", test_seq)]:
    ax.hist(frame["length"], bins=bins, weights=np.full(len(frame), 1 / len(frame)), histtype="step",
            linewidth=2, color=SPLIT_COLORS[split],
            label=f"{split} (median {int(frame['length'].median())} aa)")
ax.set_xscale("log")
ax.set_xlabel("sequence length (amino acids, log scale)")
ax.set_ylabel("share of proteins per bin")
ax.set_title("Sequence length distribution, train vs test")
ax.legend()
plt.tight_layout()
plt.show()

q1, q3 = train_seq["length"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"IQR rule: lengths above {upper_fence:.0f} aa are outliers -> {(train_seq['length'] > upper_fence).mean():.1%} of train proteins")
for limit in (512, 1024, 2048):
    print(f"train proteins longer than {limit:>4} aa: {(train_seq['length'] > limit).mean():6.1%}")
display(train_seq.nlargest(5, "length")[["EntryID", "taxonomyID", "length"]])

**Interpretation — write after running:**
- Typical length and how skewed the distribution is (median vs the longest proteins).
- Outliers: how many proteins are flagged by the IQR rule? They are real, very long proteins, not errors, so we keep them.
- Do train and test have the same length profile?
- What share would be truncated at 1024 residues (relevant for Assignment 2)?

### 3.2 Amino-acid composition

In [ ]:
composition = pd.DataFrame({
    "train": amino_acid_composition(train_seq["sequence"]),
    "test": amino_acid_composition(test_seq["sequence"]),
})
order = composition.drop(index="other").sort_values("train", ascending=False).index

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(order))
for offset, split in [(-0.2, "train"), (0.2, "test")]:
    ax.bar(x + offset, composition.loc[order, split] * 100, width=0.36, color=SPLIT_COLORS[split], label=split)
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel("share of residues (%)")
ax.set_title("Amino-acid composition, train vs test")
ax.grid(axis="x", visible=False)
ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2)
plt.tight_layout()
plt.show()

print(f"non-standard residues: train {composition.loc['other', 'train']:.4%}, test {composition.loc['other', 'test']:.4%}")
print(f"largest train/test difference for one residue: {(composition['train'] - composition['test']).abs().max():.3%} points")

**Interpretation — write after running:**
- Most and least common residues; is the composition of train and test practically identical?
- How rare are non-standard residues?

### 3.3 Species (taxonomy)

Function annotation effort is concentrated on a few model organisms. If the species mix of the test set differs from training, validation scores can be too optimistic.

In [ ]:
species_name = dict(zip(test_taxa["taxonomyID"], test_taxa["species"]))
train_taxon_share = train_tax["taxonomyID"].value_counts(normalize=True)
test_taxon_share = test_seq["taxonomyID"].dropna().astype(int).value_counts(normalize=True)

top_taxa = train_taxon_share.head(12)
labels = [f"{species_name.get(t, 'taxon')} ({t})" for t in top_taxa.index]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(np.arange(len(top_taxa))[::-1], top_taxa.to_numpy() * 100, height=0.7, color=SPLIT_COLORS["train"])
ax.set_yticks(np.arange(len(top_taxa))[::-1])
ax.set_yticklabels(labels)
for y, value in zip(np.arange(len(top_taxa))[::-1], top_taxa.to_numpy() * 100):
    ax.text(value, y, f" {value:.1f}%", va="center", fontsize=8, color=NEUTRAL)
ax.set_xlabel("share of training proteins (%)")
ax.set_title(f"The {len(top_taxa)} most common species in the training set")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

taxon_table = pd.DataFrame({"train share": train_taxon_share, "test share": test_taxon_share}).fillna(0.0)
taxon_table.insert(0, "species", [species_name.get(t, "") for t in taxon_table.index])
display(taxon_table.sort_values("train share", ascending=False).head(12))

cumulative = train_taxon_share.cumsum().to_numpy()
print(f"species in train: {len(train_taxon_share):,}; species needed to cover 80% of training proteins: {int(np.searchsorted(cumulative, 0.8) + 1)}")
train_taxa_set = set(train_taxon_share.index)
print(f"test proteins whose species never appears in train: {(~test_seq['taxonomyID'].dropna().astype(int).isin(train_taxa_set)).mean():.1%}")

**Interpretation — write after running:**
- Which organisms dominate, and how concentrated is the data?
- Is the test set dominated by the same species? What share of test proteins comes from species unseen in training?

### 3.4 Target: the three aspects

In [ ]:
aspect_summary = train_terms.groupby("aspect").agg(
    annotation_rows=("term", "size"),
    distinct_terms=("term", "nunique"),
    proteins=("EntryID", "nunique"),
).reindex(ASPECTS)
aspect_summary["share of proteins"] = aspect_summary["proteins"] / train_terms["EntryID"].nunique()
aspect_summary["rows per protein"] = aspect_summary["annotation_rows"] / aspect_summary["proteins"]
display(aspect_summary)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, column, title in zip(axes, ["annotation_rows", "distinct_terms", "proteins"],
                             ["Annotation rows", "Distinct GO terms", "Annotated proteins"]):
    ax.bar(ASPECTS, aspect_summary[column], width=0.6, color=[ASPECT_COLORS[a] for a in ASPECTS])
    for i, value in enumerate(aspect_summary[column]):
        ax.text(i, value, format_count(value), ha="center", va="bottom", fontsize=8, color=NEUTRAL)
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

# which combinations of aspects does a protein have labels for?
combos = (train_terms.drop_duplicates(["EntryID", "aspect"]).sort_values("aspect")
          .groupby("EntryID")["aspect"].agg("+".join).value_counts())
display(pd.DataFrame({"proteins": combos, "share": combos / combos.sum()}))

**Interpretation — write after running:**
- Which aspect has the most annotations and the largest vocabulary? Which is the smallest?
- What share of proteins has labels in all three aspects? (A missing aspect means "unknown", not "no function".)

### 3.5 Target: number of GO terms per protein

In [ ]:
per_protein = train_terms.groupby(["aspect", "EntryID"]).size()

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)
for ax, aspect in zip(axes, ASPECTS):
    counts = per_protein.loc[aspect]
    upper = int(max(counts.quantile(0.99), 5))
    ax.hist(counts.clip(upper=upper), bins=np.arange(0.5, upper + 1.5, 1), color=ASPECT_COLORS[aspect], rwidth=0.85)
    ax.set_title(f"{aspect}: median {counts.median():.0f}, max {counts.max()}")
    ax.set_xlabel(f"terms per protein (values above {upper} shown at {upper})")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("number of proteins")
fig.suptitle("Annotated GO terms per protein, as given in train_terms.tsv", x=0.01, ha="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

display(per_protein.groupby("aspect").describe().reindex(ASPECTS).round(2))

**Interpretation — write after running:**
- How many terms does a typical protein have in each aspect, and how heavy is the tail (compare median and max)?
- Proteins with very many terms are well-studied proteins: outliers in the target, but valid data.

### 3.6 Target: the GO hierarchy and label propagation

A specific term implies all of its ancestors (*true path rule*), and the evaluation propagates labels this way. We first check whether `train_terms.tsv` already contains the ancestors, then add them.

In [ ]:
# For each annotation: are all direct parents of the term also annotated on that protein?
n_checked = n_complete = 0
for protein_terms in train_terms.groupby("EntryID")["term"].agg(set):
    canonical = {go.canonical(t) for t in protein_terms}
    for term in canonical:
        parents = go.parents.get(term, set())
        if parents:
            n_checked += 1
            n_complete += parents <= canonical
print(f"annotations whose direct parents are all present: {n_complete / max(n_checked, 1):.1%}")

start = time.time()
train_prop = propagate_annotations(train_terms, go)
print(f"propagation took {time.time() - start:.1f} s")

before = train_terms.groupby("EntryID")["term"].nunique()
after = train_prop.groupby("EntryID")["term"].nunique()
display(pd.DataFrame({
    "as given": [len(train_terms), train_terms["term"].nunique(), before.mean(), before.median()],
    "after propagation": [len(train_prop), train_prop["term"].nunique(), after.mean(), after.median()],
}, index=["annotation rows", "distinct terms", "mean terms per protein", "median terms per protein"]).round(1))

depth_of = {t: go.depth(t) for t in train_terms["term"].unique()}
depth_frame = train_terms.assign(depth=train_terms["term"].map(depth_of))

fig, ax = plt.subplots(figsize=(8, 3.4))
for aspect in ASPECTS:
    shares = depth_frame.loc[depth_frame["aspect"] == aspect, "depth"].value_counts(normalize=True).sort_index()
    ax.plot(shares.index, shares.to_numpy() * 100, marker="o", markersize=5, color=ASPECT_COLORS[aspect],
            label=f"{aspect} (mean depth {depth_frame.loc[depth_frame['aspect'] == aspect, 'depth'].mean():.1f})")
ax.set_xlabel("depth of the annotated term (shortest path to the root; root = 0)")
ax.set_ylabel("share of annotations (%)")
ax.set_title("How specific are the given annotations?")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation — write after running:**
- Is the training file already propagated (close to 100% above) or does it list only the most specific terms?
- How much do the number of rows and of distinct terms grow after propagation?
- At which depth do most annotations sit? Deeper terms are more specific and harder to predict.

From here on we use the **propagated** labels (`train_prop`), because that is what the evaluation uses.

### 3.7 Target balance: a very long tail

In a multi-label problem "class balance" means: how many proteins does each GO term have?

In [ ]:
n_labelled = train_prop["EntryID"].nunique()
term_counts = train_prop.groupby(["aspect", "term"]).size().rename("proteins").reset_index()
term_counts = term_counts.sort_values(["aspect", "proteins"], ascending=[True, False])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for aspect in ASPECTS:
    counts = term_counts.loc[term_counts["aspect"] == aspect, "proteins"].to_numpy()
    rank = np.arange(1, len(counts) + 1)
    axes[0].plot(rank, counts, color=ASPECT_COLORS[aspect], label=f"{aspect} ({len(counts):,} terms)")
    axes[1].plot(rank, np.cumsum(counts) / counts.sum() * 100, color=ASPECT_COLORS[aspect], label=aspect)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("term rank within aspect (log scale)")
axes[0].set_ylabel("proteins annotated with the term (log scale)")
axes[0].set_title("Rank vs frequency of GO terms")
axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_xlabel("number of most frequent terms kept (log scale)")
axes[1].set_ylabel("share of annotation rows covered (%)")
axes[1].set_title("Coverage of the top-N terms")
axes[1].legend()
plt.tight_layout()
plt.show()

tail = term_counts.groupby("aspect")["proteins"].agg(
    terms="size",
    median_proteins="median",
    terms_under_10=lambda s: (s < 10).sum(),
    terms_under_50=lambda s: (s < 50).sum(),
).reindex(ASPECTS)
tail["share under 10"] = tail["terms_under_10"] / tail["terms"]
display(tail)

top_terms = term_counts.groupby("aspect").head(8).copy()
top_terms["name"] = top_terms["term"].map(go.name)
top_terms["share of proteins"] = top_terms["proteins"] / n_labelled
top_terms["IA"] = top_terms["term"].map(ia)
display(top_terms[["aspect", "term", "name", "proteins", "share of proteins", "IA"]].reset_index(drop=True))

**Interpretation — write after running:**
- How imbalanced is the target? How many terms have fewer than 10 training proteins, and what does that mean for learning them?
- How many top terms are needed to cover most of the annotation rows in each aspect?
- The most frequent terms are generic terms near the root; look at their IA values.

### 3.8 Information accretion (IA) weights

The metric weights every term by its IA (section 1.3), so it matters how IA relates to term frequency.

In [ ]:
ia_frame = term_counts.assign(IA=term_counts["term"].map(ia).fillna(0.0))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ia_bins = np.linspace(0, max(ia_frame["IA"].max(), 1e-9), 40)
for aspect in ASPECTS:
    subset = ia_frame[ia_frame["aspect"] == aspect]
    axes[0].hist(subset["IA"], bins=ia_bins, histtype="step", linewidth=2, color=ASPECT_COLORS[aspect],
                 weights=np.full(len(subset), 1 / max(len(subset), 1)),
                 label=f"{aspect} (median {subset['IA'].median():.2f})")
    axes[1].scatter(subset["proteins"], subset["IA"], s=14, alpha=0.45, color=ASPECT_COLORS[aspect],
                    edgecolors="none", label=aspect)
axes[0].set_xlabel("IA weight")
axes[0].set_ylabel("share of terms per bin")
axes[0].set_title("Distribution of IA weights (terms used in train)")
axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_xlabel("proteins annotated with the term (log scale)")
axes[1].set_ylabel("IA weight")
axes[1].set_title("IA weight vs term frequency")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"terms with IA = 0: {(ia_frame['IA'] == 0).mean():.1%} of the terms used in train")
print("Spearman correlation between term frequency and IA:",
      round(ia_frame[["proteins", "IA"]].corr(method="spearman").iloc[0, 1], 3))
print("IA of the three root terms:", {a: float(ia.get(t, np.nan)) for a, t in ROOT_TERMS.items()})

**Interpretation — write after running:**
- How are IA weights distributed, and how strongly are they related to term frequency?
- Why does this make the task hard? (The terms that are easiest to predict are worth the least.)

### 3.9 Relationships between the inputs and the target

Do the two inputs we have, sequence length and species, tell us anything about the labels?

In [ ]:
terms_wide = train_prop.groupby(["EntryID", "aspect"]).size().unstack(fill_value=0).reindex(columns=ASPECTS, fill_value=0)
relation = train_seq.drop_duplicates("EntryID").set_index("EntryID")[["length"]].join(terms_wide, how="inner")
relation["all aspects"] = relation[ASPECTS].sum(axis=1)
relation["length_bin"] = pd.qcut(relation["length"], 10, duplicates="drop")
by_length = relation.groupby("length_bin", observed=True)

fig, ax = plt.subplots(figsize=(8, 3.6))
for aspect in ASPECTS:
    # mean over the proteins that are annotated in this aspect
    means = by_length[aspect].apply(lambda s: s[s > 0].mean())
    ax.plot(by_length["length"].median().to_numpy(), means.to_numpy(), marker="o", markersize=5,
            color=ASPECT_COLORS[aspect], label=aspect)
ax.set_xscale("log")
ax.set_xlabel("sequence length (median of each length decile, log scale)")
ax.set_ylabel("mean number of GO terms (propagated)")
ax.set_title("Sequence length vs number of annotated terms")
ax.legend()
plt.tight_layout()
plt.show()

print("Spearman correlation between sequence length and number of terms:",
      round(relation[["length", "all aspects"]].corr(method="spearman").iloc[0, 1], 3))

**Interpretation — write after running:**
- Do longer proteins have more annotated terms? How strong is the relationship (Spearman correlation)?
- Is length alone a useful predictor, or do we need the content of the sequence?

In [ ]:
by_species = relation.join(train_tax.drop_duplicates("EntryID").set_index("EntryID")["taxonomyID"], how="inner")
grouped = by_species.groupby("taxonomyID")
species_stats = pd.DataFrame({
    "proteins": grouped.size(),
    "mean_terms": grouped["all aspects"].mean(),
    "share with all three aspects": (by_species[ASPECTS] > 0).all(axis=1).groupby(by_species["taxonomyID"]).mean(),
}).sort_values("proteins", ascending=False).head(10)
species_labels = [f"{species_name.get(t, 'taxon')} ({t})" for t in species_stats.index]

fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(species_stats))[::-1]
ax.barh(y, species_stats["mean_terms"], height=0.7, color=NEUTRAL)
for yi, value in zip(y, species_stats["mean_terms"]):
    ax.text(value, yi, f" {value:.1f}", va="center", fontsize=8, color=NEUTRAL)
ax.set_yticks(y)
ax.set_yticklabels(species_labels)
ax.set_xlabel("mean number of GO terms per protein (propagated)")
ax.set_title(f"Mean number of GO terms per protein in the {len(species_stats)} largest species")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()
display(species_stats)

**Interpretation — write after running:**
- Which species are annotated most thoroughly (most terms per protein, all three aspects present)?
- What does this imply for a model evaluated on proteins from many species? Could species be a useful feature (Assignment 2)?

### 3.10 EDA summary

> **TODO — summarise the main findings in your own words (5–8 bullet points)** with the actual numbers from the cells above: size of the data, multi-label target and number of labels, missing aspects, propagation, long tail, IA weighting, train/test differences (length, species), and what each finding implies for preprocessing and modelling.

## 4. Preprocessing

Each step below is motivated by a finding from sections 2–3. **TODO:** after running, check every "why" against your own numbers and adjust the wording where your data says something different.

| Step | What we do | Why |
|---|---|---|
| Cleaning | keep one row per protein id; keep proteins that have both a sequence and labels; map alternative GO ids to primary ids | duplicates would leak between train and validation; unlabeled proteins cannot be used for supervised training |
| Non-standard residues | kept as they are | they are rare (see the share printed in section 3.2) and simply form their own rare k-mers |
| Outliers | very long proteins and heavily annotated proteins are kept | they are real biology, not measurement errors (sections 3.1, 3.5) |
| Missing values / imputation | nothing is imputed | see the counts in section 2.3; the inputs are sequences, not numeric columns that could be filled, and a missing aspect means "unknown", so inventing labels would be wrong |
| Target encoding | propagate labels up the GO graph, then encode as a sparse multi-hot matrix | the evaluation uses propagated labels (section 3.6); thousands of labels need a sparse format |
| Label space | predict only the most frequent terms per aspect | most terms have too few examples to learn (section 3.7); recall is still measured against all true terms |
| Split | 20% held-out validation, `random_state=42` | test labels are not public; stratification is not possible with thousands of labels per sample |
| Feature encoding | overlapping 3-mers, TF-IDF weighted | turns variable-length strings into fixed-length numeric vectors |
| Scaling | sublinear term frequency and L2 normalisation of every vector | removes the effect of sequence length (section 3.1), so long proteins do not dominate |

The featuriser is **fitted on the training split only**, to avoid leaking validation information.

In [ ]:
# ---- configuration -------------------------------------------------------
LABELS_PER_ASPECT = {"BPO": 600, "CCO": 200, "MFO": 300}   # most frequent terms kept per aspect
VAL_FRACTION = 0.2
KMER_K = 3
KNN_K = 10
MLP_HIDDEN = 512
MLP_EPOCHS = 15
MAX_PROTEINS = None   # e.g. 5000 for a quick debugging run; None = use everything
# --------------------------------------------------------------------------

# cleaning: one sequence per id, only proteins that have a sequence and at least one label
sequences = train_seq.drop_duplicates("EntryID").set_index("EntryID")["sequence"]
sequences = sequences[sequences.str.len() > 0]
protein_ids = np.array(sorted(set(train_prop["EntryID"]) & set(sequences.index)))
print(f"proteins after cleaning: {len(protein_ids):,} (of {train_seq['EntryID'].nunique():,} in the FASTA file)")
if MAX_PROTEINS is not None and len(protein_ids) > MAX_PROTEINS:
    protein_ids = np.random.default_rng(SEED).choice(protein_ids, MAX_PROTEINS, replace=False)

# split
train_ids, val_ids = train_test_split(protein_ids, test_size=VAL_FRACTION, random_state=SEED)
print(f"train proteins: {len(train_ids):,}   validation proteins: {len(val_ids):,}")

# target encoding: sparse multi-hot matrices over every term seen in training
all_terms = sorted(train_prop["term"].unique())
term_aspect = dict(zip(train_prop["term"], train_prop["aspect"]))
Y_train_all = build_label_matrix(train_prop, train_ids, all_terms)
Y_val_all = build_label_matrix(train_prop, val_ids, all_terms)
print(f"label matrix (train): {Y_train_all.shape[0]:,} x {Y_train_all.shape[1]:,}, "
      f"{Y_train_all.nnz / (Y_train_all.shape[0] * Y_train_all.shape[1]):.3%} of the cells are 1")

In [ ]:
# label space: the most frequent terms of each aspect, counted on the training split only
train_counts = np.asarray(Y_train_all.sum(axis=0)).ravel()
all_aspects = np.array([term_aspect[t] for t in all_terms])

label_cols = []
for aspect in ASPECTS:
    cols = np.where((all_aspects == aspect) & (train_counts > 0))[0]
    label_cols.extend(cols[np.argsort(-train_counts[cols], kind="stable")][: LABELS_PER_ASPECT[aspect]])
label_cols = np.array(label_cols)
label_terms = [all_terms[j] for j in label_cols]
label_aspects = all_aspects[label_cols]

Y_train = Y_train_all[:, label_cols]
Y_val = Y_val_all[:, label_cols]

ia_all = ia.reindex(all_terms).fillna(0.0).to_numpy()
val_counts = np.asarray(Y_val_all.sum(axis=0)).ravel()
in_labels = np.zeros(len(all_terms), dtype=bool)
in_labels[label_cols] = True
coverage = []
for aspect in ASPECTS:
    mask = all_aspects == aspect
    coverage.append({
        "aspect": aspect,
        "terms in train": int(mask.sum()),
        "terms we predict": int((mask & in_labels).sum()),
        "least frequent kept term (train proteins)": int(train_counts[mask & in_labels].min()),
        "validation annotations covered": val_counts[mask & in_labels].sum() / max(val_counts[mask].sum(), 1),
        "IA-weighted coverage": (val_counts * ia_all)[mask & in_labels].sum() / max((val_counts * ia_all)[mask].sum(), 1e-12),
    })
display(pd.DataFrame(coverage).set_index("aspect"))

The table shows the price of truncating the label space: plain coverage is high because frequent terms dominate the annotation rows, but **IA-weighted coverage is lower**, since the rare terms we dropped are exactly the ones with high weight.

> **TODO:** quote the two coverage numbers per aspect from your run and state whether the trade-off is acceptable for a baseline.

In [ ]:
# feature encoding and scaling: k-mer TF-IDF, fitted on the training split only
start = time.time()
featurizer = KmerFeaturizer(k=KMER_K)
X_train = featurizer.fit_transform(sequences.loc[train_ids])
X_val = featurizer.transform(sequences.loc[val_ids])
row_norms = np.sqrt(np.asarray(X_train.multiply(X_train).sum(axis=1)).ravel())
print(f"k-mer TF-IDF features: {X_train.shape[1]:,} columns, "
      f"{X_train.nnz / X_train.shape[0]:.0f} non-zeros per protein on average ({time.time() - start:.1f} s)")
print(f"L2 norm of the feature vectors: min {row_norms.min():.3f}, max {row_norms.max():.3f} (all scaled to length 1)")

## 5. Baseline models

All models are trained on the training split and scored on the validation split with the competition metric (section 1.3).

| Baseline | Uses the sequence? | Idea |
|---|---|---|
| Naive | no | every protein gets the same scores: the training frequency of each term |
| k-NN on k-mers | yes | copy labels from the most similar training sequences (homology transfer) |
| MLP on k-mers | yes | one-hidden-layer neural network, one sigmoid output per term |

For the MLP we also report the scores after **hierarchy propagation** (a parent gets at least the score of its highest-scoring child). The naive and k-NN scores respect the hierarchy by construction.

### 5.1 Evaluation function and sanity checks

Predicting the exact truth for every kept term gives the **upper bound** our label space allows; random scores show the floor.

In [ ]:
results, curves = [], {}


def evaluate(name, scores, keep=True):
    res = evaluate_predictions(Y_val_all, all_terms, scores, label_terms, ia, term_aspect)
    if keep:
        results.append(results_row(name, res))
        curves[name] = res
    print(f"{name:<42} " + "  ".join(f"{a} {res[a]['fmax']:.4f}" for a in ASPECTS) + f"   mean {res['mean_fmax']:.4f}")
    return res


oracle = evaluate("Upper bound (perfect on kept terms)", Y_val.toarray(), keep=False)
random_scores = np.random.default_rng(SEED).random(Y_val.shape, dtype=np.float32)
_ = evaluate("Random scores", random_scores, keep=False)

### 5.2 Naive baseline (term frequency)

In [ ]:
naive = NaiveFrequencyBaseline(label_aspects).fit(X_train, Y_train)
scores_naive = naive.predict(X_val)
_ = evaluate("Naive (term frequency)", scores_naive)

### 5.3 k-nearest neighbours on k-mer TF-IDF

In [ ]:
start = time.time()
knn = KNNBaseline(k=KNN_K).fit(X_train, Y_train)
scores_knn = knn.predict(X_val)
print(f"k-NN prediction took {time.time() - start:.1f} s")
_ = evaluate(f"k-NN (k={KNN_K})", scores_knn)

# k-NN averages labels that are already propagated, so its scores respect the hierarchy by construction:
change = np.abs(propagate_scores(scores_knn, label_terms, go) - scores_knn).max()
print(f"largest score change if we propagate the k-NN scores: {change:.2e}")

### 5.4 MLP on k-mer TF-IDF

In [ ]:
start = time.time()
mlp = MLPBaseline(hidden_units=MLP_HIDDEN, epochs=MLP_EPOCHS, random_state=SEED).fit(X_train, Y_train)
print(f"MLP training took {time.time() - start:.1f} s")
scores_mlp = mlp.predict(X_val)
_ = evaluate("MLP", scores_mlp)
scores_mlp_prop = propagate_scores(scores_mlp, label_terms, go)
_ = evaluate("MLP + propagation", scores_mlp_prop)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(mlp.loss_curve) + 1), mlp.loss_curve, color=NEUTRAL, marker="o", markersize=4)
ax.set_xlabel("epoch")
ax.set_ylabel("training loss (binary cross-entropy)")
ax.set_title("MLP training loss")
plt.tight_layout()
plt.show()

**Interpretation — write after running:**
- Is the training loss still decreasing after the last epoch? (If yes, more epochs might help; watch the validation score, not only the loss.)

### 5.5 Results on the validation split

In [ ]:
results_table = pd.DataFrame(results).set_index("model")
display(results_table)
print(f"upper bound with this label space: mean F-max {oracle['mean_fmax']:.4f}")

panels = [f"Fmax {a}" for a in ASPECTS] + ["Fmax mean"]
panel_colors = [ASPECT_COLORS[a] for a in ASPECTS] + [NEUTRAL]
fig, axes = plt.subplots(1, 4, figsize=(13, 0.5 * len(results_table) + 1.6), sharey=True)
y = np.arange(len(results_table))[::-1]
for ax, column, color in zip(axes, panels, panel_colors):
    ax.barh(y, results_table[column], height=0.65, color=color)
    for yi, value in zip(y, results_table[column]):
        ax.text(value, yi, f" {value:.3f}", va="center", fontsize=8, color=NEUTRAL)
    ax.set_title(column.replace("Fmax ", "F-max, "))
    ax.set_xlim(0, max(results_table[panels].to_numpy().max() * 1.25, 0.05))
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(y)
axes[0].set_yticklabels(results_table.index)
fig.suptitle("Validation F-max (IA-weighted) of the baselines", x=0.01, ha="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation — write after running:**
- Rank the baselines. How much do the sequence-based models gain over the naive baseline, and in which aspect?
- Does hierarchy propagation help the MLP?
- Which aspect is hardest? Relate this to the EDA (vocabulary size, depth, long tail).

In [ ]:
# precision-recall curves of the three model families (best variant of each)
def best_variant(prefix):
    names = [n for n in curves if n.startswith(prefix)]
    return max(names, key=lambda n: curves[n]["mean_fmax"])

shown = [best_variant("Naive"), best_variant("k-NN"), best_variant("MLP")]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), sharex=True, sharey=True)
for ax, aspect in zip(axes, ASPECTS):
    for name, (color, style) in zip(shown, MODEL_STYLES):
        curve = curves[name][aspect]["curve"]
        curve = curve[(curve["precision"] > 0) | (curve["recall"] > 0)]
        ax.plot(curve["recall"], curve["precision"], color=color, linestyle=style,
                label=f"{name} (F-max {curves[name][aspect]['fmax']:.3f})")
    ax.set_title(f"{aspect} — {ASPECT_FULL_NAMES[aspect]}")
    ax.set_xlabel("weighted recall")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8)
axes[0].set_ylabel("weighted precision")
plt.tight_layout()
plt.show()

**Interpretation — write after running:**
- How do precision and recall trade off for each model? Where does each curve end (maximum reachable recall)?
- How far is the best baseline from the upper bound of this label space, and what limits it (k-mers ignore long-range order; rare terms are dropped; the random split is optimistic because similar sequences can fall on both sides)?

## 6. Kaggle submission

The competition expects a tab-separated file without header: `protein id`, `GO term`, `score` in (0, 1]. We use the best baseline on validation, propagate its scores and keep the highest scoring terms per protein. The file is large, so it is **not committed** (see `.gitignore`).

In [ ]:
MAKE_SUBMISSION = True
SUBMISSION_PATH = Path("submission.tsv")
SUBMISSION_TOP_K = 40      # terms written per protein
SUBMISSION_MODEL = None    # None = best baseline on validation; or force "k-NN" / "MLP"
CHUNK = 5000               # note: k-NN compares every test protein with every training protein,
                           # so on the full test set it is much slower than the MLP

models = {"k-NN": knn, "MLP": mlp}
best_family = SUBMISSION_MODEL or max(models, key=lambda fam: curves[best_variant(fam)]["mean_fmax"])
print("model used for the submission:", best_variant(best_family))

if MAKE_SUBMISSION:
    start = time.time()
    targets = test_seq.drop_duplicates("EntryID")
    n_lines = 0
    for begin in range(0, len(targets), CHUNK):
        chunk = targets.iloc[begin:begin + CHUNK]
        scores = models[best_family].predict(featurizer.transform(chunk["sequence"]))
        scores = propagate_scores(scores, label_terms, go)
        n_lines += write_submission(SUBMISSION_PATH, chunk["EntryID"].to_numpy(), scores, label_terms,
                                    top_k=SUBMISSION_TOP_K, append=begin > 0)
        if (begin // CHUNK) % 5 == 0:
            print(f"  {min(begin + CHUNK, len(targets)):>7,} / {len(targets):,} proteins ({time.time() - start:.0f} s)")
    print(f"wrote {n_lines:,} lines for {len(targets):,} proteins to {SUBMISSION_PATH} "
          f"({SUBMISSION_PATH.stat().st_size / 1e6:.1f} MB, {time.time() - start:.0f} s)")
    display(pd.read_csv(SUBMISSION_PATH, sep="\t", header=None, names=["protein", "term", "score"], nrows=5))

### Submitting

The competition is closed, so this is a **late submission** (scored, but not placed on the leaderboard). Set `SUBMIT_TO_KAGGLE = True` to upload the file with the Kaggle API, or upload `submission.tsv` by hand on the competition page (*Late Submission / Submit Prediction*). Then take a screenshot of the score and save it in `submissions/`.

In [ ]:
SUBMIT_TO_KAGGLE = False
SUBMISSION_MESSAGE = f"Assignment 1 baseline: {best_variant(best_family)}"

if SUBMIT_TO_KAGGLE and SUBMISSION_PATH.exists():
    ensure_kaggle_credentials()
    done = subprocess.run(["kaggle", "competitions", "submit", "-c", COMPETITION, "-f", str(SUBMISSION_PATH),
                           "-m", SUBMISSION_MESSAGE], capture_output=True, text=True)
    print(done.stdout or done.stderr)
    listing = subprocess.run(["kaggle", "competitions", "submissions", "-c", COMPETITION],
                             capture_output=True, text=True)
    print(listing.stdout or listing.stderr)

> **TODO — fill in after submitting.**
>
> | Date | Model | Validation F-max (mean) | Kaggle public score | Kaggle private score | Screenshot |
> |---|---|---|---|---|---|
> | TODO | TODO | TODO | TODO | TODO | `submissions/TODO.png` |
>
> **If the late submission is not accepted:** state it here, and report the held-out evaluation instead: 20% of the training proteins held out with `random_state=42` (section 4), scored with the competition metric (section 5.5). Stratification is not applicable because every protein has many labels.
>
> **Gap to the public leaderboard (one sentence):** compare your score with the top public scores of the competition and with public baseline notebooks, and say why the gap exists (e.g. top solutions use protein language model embeddings and much larger label spaces).

## 7. Conclusion

> **TODO — write in your own words.**
>
> **Key findings from the data:** (3–4 sentences from section 3.10)
>
> **Baseline results:** (best baseline, its mean F-max, comparison with the naive baseline, Kaggle score)
>
> **Limitations:** random split is optimistic; only the top terms are predicted; k-mers are a shallow representation.
>
> **Next steps for Assignment 2:** e.g. protein language model embeddings (ESM-2 / ProtT5) as features, a larger label space, a homology-aware validation split, combining sequence similarity with the neural model, species as an extra feature.